In [1]:
# =============================================================================
# STEP 02.2 — CROSS-MODAL PAIRING STRATEGY COMPARISON
# =============================================================================
#
# Objective:
# Compare scientifically plausible strategies for constructing relationships
# between independently learned modality representations.
#
# Analysis Level:
# Knowledge-Level Cross-Modal Learning
#
# Important:
# - No patient-level matching
# - No shared patient identity assumed
# - Diagnostic class is used as semantic supervision
# - Comparisons are performed between representation-level strategies
#
# Candidate Strategies:
# 1. Random Same-Class Pairing
# 2. Balanced Class-Conditional Pairing
# 3. Centroid / Prototype-Based Pairing
# 4. Similarity / Distribution-Aware Pairing
#
# =============================================================================

print("=" * 100)
print("STEP 02.2 — CROSS-MODAL PAIRING STRATEGY COMPARISON")
print("=" * 100)

print("\nAnalysis Type:")
print("Knowledge-Level Cross-Modal Learning")

print("\nPatient-Level Matching:")
print("NOT USED")

print("\nPrimary Objective:")
print(
    "Compare possible strategies for constructing scientifically valid "
    "cross-modal relationships between independently learned 32D embeddings."
)

print("\nCandidate Strategies:")
print("1. Random Same-Class Pairing")
print("2. Balanced Class-Conditional Pairing")
print("3. Centroid / Prototype-Based Pairing")
print("4. Similarity / Distribution-Aware Pairing")

print("\n" + "=" * 100)
print("INITIALIZATION COMPLETED")
print("=" * 100)

STEP 02.2 — CROSS-MODAL PAIRING STRATEGY COMPARISON

Analysis Type:
Knowledge-Level Cross-Modal Learning

Patient-Level Matching:
NOT USED

Primary Objective:
Compare possible strategies for constructing scientifically valid cross-modal relationships between independently learned 32D embeddings.

Candidate Strategies:
1. Random Same-Class Pairing
2. Balanced Class-Conditional Pairing
3. Centroid / Prototype-Based Pairing
4. Similarity / Distribution-Aware Pairing

INITIALIZATION COMPLETED


In [2]:
# =============================================================================
# CELL 2 — IMPORT LIBRARIES AND RESOLVE PROJECT PATHS
# =============================================================================

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances

print("=" * 100)
print("IMPORTS AND PROJECT PATH CONFIGURATION")
print("=" * 100)

# -------------------------------------------------------------------------
# Resolve Project Root
# -------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "GAN-Enhanced-Multimodal-ASD-Screening":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate project root: GAN-Enhanced-Multimodal-ASD-Screening"
        )
    PROJECT_ROOT = PROJECT_ROOT.parent


PAIRING_DIR = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
)

REPORT_DIR = PAIRING_DIR / "reports"
FIGURE_DIR = PAIRING_DIR / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nProject Root:\n{PROJECT_ROOT}")
print(f"\nPairing Directory:\n{PAIRING_DIR}")
print(f"\nReport Directory:\n{REPORT_DIR}")

print("\n" + "=" * 100)
print("PATH CONFIGURATION COMPLETED")
print("=" * 100)

IMPORTS AND PROJECT PATH CONFIGURATION

Project Root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Pairing Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing

Report Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports

PATH CONFIGURATION COMPLETED


In [3]:
# =============================================================================
# CELL 3 — LOAD FROZEN EMBEDDINGS AND LABELS
# =============================================================================

print("=" * 100)
print("LOADING FROZEN REPRESENTATIONS FOR STRATEGY COMPARISON")
print("=" * 100)

FUSION_DIR = PROJECT_ROOT / "05_multimodal_fusion"

# -------------------------------------------------------------------------
# File Paths
# -------------------------------------------------------------------------

GENETIC_EMBEDDING_PATH = FUSION_DIR / "genetic_32d_normalized.npy"
GENETIC_TARGET_PATH = FUSION_DIR / "genetic_targets.npy"

BEHAVIORAL_EMBEDDING_PATH = FUSION_DIR / "behavioral_32d_normalized.npy"
BEHAVIORAL_TARGET_PATH = FUSION_DIR / "behavioral_targets.npy"

ENVIRONMENTAL_EMBEDDING_PATH = FUSION_DIR / "environmental_32d_normalized.npy"
ENVIRONMENTAL_TARGET_PATH = FUSION_DIR / "environmental_targets.npy"


# -------------------------------------------------------------------------
# Load Data
# -------------------------------------------------------------------------

genetic_embeddings = np.load(GENETIC_EMBEDDING_PATH)
genetic_labels = np.load(GENETIC_TARGET_PATH)

behavioral_embeddings = np.load(BEHAVIORAL_EMBEDDING_PATH)
behavioral_labels = np.load(BEHAVIORAL_TARGET_PATH)

environmental_embeddings = np.load(ENVIRONMENTAL_EMBEDDING_PATH)
environmental_labels = np.load(ENVIRONMENTAL_TARGET_PATH)


modalities = {
    "Genetic": {
        "embeddings": genetic_embeddings,
        "labels": genetic_labels
    },
    "Behavioral": {
        "embeddings": behavioral_embeddings,
        "labels": behavioral_labels
    },
    "Environmental": {
        "embeddings": environmental_embeddings,
        "labels": environmental_labels
    }
}


# -------------------------------------------------------------------------
# Display Summary
# -------------------------------------------------------------------------

for modality, data in modalities.items():

    embeddings = data["embeddings"]
    labels = data["labels"]

    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{modality.upper()}")
    print("-" * 70)
    print(f"Embedding Shape : {embeddings.shape}")
    print(f"Label Shape     : {labels.shape}")
    print(f"Class Counts    : {dict(zip(unique, counts))}")

print("\n" + "=" * 100)
print("FROZEN REPRESENTATIONS LOADED SUCCESSFULLY")
print("=" * 100)

LOADING FROZEN REPRESENTATIONS FOR STRATEGY COMPARISON

GENETIC
----------------------------------------------------------------------
Embedding Shape : (116, 32)
Label Shape     : (116,)
Class Counts    : {np.int64(0): np.int64(29), np.int64(1): np.int64(87)}

BEHAVIORAL
----------------------------------------------------------------------
Embedding Shape : (880, 32)
Label Shape     : (880,)
Class Counts    : {np.int64(0): np.int64(439), np.int64(1): np.int64(441)}

ENVIRONMENTAL
----------------------------------------------------------------------
Embedding Shape : (47768, 32)
Label Shape     : (47768,)
Class Counts    : {np.int64(0): np.int64(45650), np.int64(1): np.int64(2118)}

FROZEN REPRESENTATIONS LOADED SUCCESSFULLY


In [4]:
# =============================================================================
# CELL 4 — DATASET CHARACTERISTICS RELEVANT TO PAIRING
# =============================================================================

print("=" * 100)
print("DATASET CHARACTERISTICS RELEVANT TO PAIRING STRATEGY")
print("=" * 100)

dataset_analysis = []

for modality, data in modalities.items():

    labels = data["labels"]

    unique, counts = np.unique(labels, return_counts=True)

    class_counts = dict(zip(unique, counts))

    count_class_0 = class_counts.get(0, 0)
    count_class_1 = class_counts.get(1, 0)

    total_samples = len(labels)

    minority_count = min(count_class_0, count_class_1)
    majority_count = max(count_class_0, count_class_1)

    imbalance_ratio = (
        majority_count / minority_count
        if minority_count > 0
        else np.inf
    )

    dataset_analysis.append({
        "Modality": modality,
        "Total_Samples": total_samples,
        "NON_ASD_Count": count_class_0,
        "ASD_Count": count_class_1,
        "NON_ASD_Percentage": round(
            count_class_0 / total_samples * 100, 2
        ),
        "ASD_Percentage": round(
            count_class_1 / total_samples * 100, 2
        ),
        "Imbalance_Ratio": round(imbalance_ratio, 4)
    })


dataset_analysis_df = pd.DataFrame(dataset_analysis)

display(dataset_analysis_df)

print("\n" + "=" * 100)
print("KEY PAIRING IMPLICATIONS")
print("=" * 100)

print("""
1. Dataset sizes differ substantially across modalities.

2. Environmental data is much larger than Genetic and Behavioral data.

3. Class distributions are not consistent across modalities.

4. Uncontrolled random pairing could cause:
   - Environmental modality domination
   - Majority class domination
   - Unequal knowledge contribution

5. Therefore, a scientifically valid strategy should explicitly control:
   - Modality contribution
   - Class contribution
   - Sample reuse
   - Pair counts
""")

print("=" * 100)

DATASET CHARACTERISTICS RELEVANT TO PAIRING STRATEGY


,Modality,Total_Samples,NON_ASD_Count,ASD_Count,NON_ASD_Percentage,ASD_Percentage,Imbalance_Ratio
0,Genetic,116,29,87,25.00,75.00,3.0000
1,Behavioral,880,439,441,49.89,50.11,1.0046
2,Environmental,47768,45650,2118,95.57,4.43,21.5534



KEY PAIRING IMPLICATIONS

1. Dataset sizes differ substantially across modalities.

2. Environmental data is much larger than Genetic and Behavioral data.

3. Class distributions are not consistent across modalities.

4. Uncontrolled random pairing could cause:
   - Environmental modality domination
   - Majority class domination
   - Unequal knowledge contribution

5. Therefore, a scientifically valid strategy should explicitly control:
   - Modality contribution
   - Class contribution
   - Sample reuse
   - Pair counts



In [5]:
# =============================================================================
# CELL 5 — DEFINE CANDIDATE PAIRING STRATEGIES
# =============================================================================

print("=" * 100)
print("CANDIDATE CROSS-MODAL PAIRING STRATEGIES")
print("=" * 100)

strategy_definitions = pd.DataFrame([

    {
        "Strategy": "Random Same-Class Pairing",
        "Core Principle":
            "Randomly pair samples from different modalities sharing the same class",
        "Patient Matching":
            "No",
        "Class Supervision":
            "Yes",
        "Modality Balance":
            "Not Guaranteed",
        "Class Balance":
            "Not Guaranteed",
        "Scalability":
            "High"
    },

    {
        "Strategy": "Balanced Class-Conditional Pairing",
        "Core Principle":
            "Construct controlled same-class and different-class relationships "
            "with balanced modality and class contributions",
        "Patient Matching":
            "No",
        "Class Supervision":
            "Yes",
        "Modality Balance":
            "Controlled",
        "Class Balance":
            "Controlled",
        "Scalability":
            "High"
    },

    {
        "Strategy": "Centroid / Prototype Pairing",
        "Core Principle":
            "Represent each modality-class group using an aggregate centroid "
            "and compare prototypes",
        "Patient Matching":
            "No",
        "Class Supervision":
            "Yes",
        "Modality Balance":
            "Naturally Controlled",
        "Class Balance":
            "Naturally Controlled",
        "Scalability":
            "Very High"
    },

    {
        "Strategy": "Similarity / Distribution-Aware Pairing",
        "Core Principle":
            "Pair representations based on existing geometric similarity "
            "or distributional proximity",
        "Patient Matching":
            "No",
        "Class Supervision":
            "Optional / Combined",
        "Modality Balance":
            "Requires Control",
        "Class Balance":
            "Requires Control",
        "Scalability":
            "Moderate"
    }

])

display(strategy_definitions)

print("\n" + "=" * 100)
print("IMPORTANT")
print("=" * 100)

print("""
At this stage, none of the strategies is automatically selected.

The next cells evaluate their suitability based on:

1. Dataset imbalance
2. Knowledge diversity
3. Risk of sampling bias
4. Dependency on natural embedding alignment
5. Compatibility with knowledge-level learning
6. Scalability
7. Reproducibility
""")

CANDIDATE CROSS-MODAL PAIRING STRATEGIES


,Strategy,Core Principle,Patient Matching,Class Supervision,Modality Balance,Class Balance,Scalability
0,Random Same-Class Pairing,Randomly pair samples from different modalitie...,No,Yes,Not Guaranteed,Not Guaranteed,High
1,Balanced Class-Conditional Pairing,Construct controlled same-class and different-...,No,Yes,Controlled,Controlled,High
2,Centroid / Prototype Pairing,Represent each modality-class group using an a...,No,Yes,Naturally Controlled,Naturally Controlled,Very High
3,Similarity / Distribution-Aware Pairing,Pair representations based on existing geometr...,No,Optional / Combined,Requires Control,Requires Control,Moderate



IMPORTANT

At this stage, none of the strategies is automatically selected.

The next cells evaluate their suitability based on:

1. Dataset imbalance
2. Knowledge diversity
3. Risk of sampling bias
4. Dependency on natural embedding alignment
5. Compatibility with knowledge-level learning
6. Scalability
7. Reproducibility



In [6]:
# =============================================================================
# CELL 6 — ANALYZE RANDOM SAME-CLASS PAIRING
# =============================================================================

print("=" * 100)
print("STRATEGY A — RANDOM SAME-CLASS PAIRING ANALYSIS")
print("=" * 100)

random_pairing_analysis = []

for modality, data in modalities.items():

    labels = data["labels"]

    unique, counts = np.unique(labels, return_counts=True)

    class_counts = dict(zip(unique, counts))

    random_pairing_analysis.append({
        "Modality": modality,
        "Total_Samples": len(labels),
        "NON_ASD_Available": class_counts.get(0, 0),
        "ASD_Available": class_counts.get(1, 0)
    })


random_pairing_df = pd.DataFrame(random_pairing_analysis)

display(random_pairing_df)

print("\n" + "-" * 100)
print("SCIENTIFIC ASSESSMENT")
print("-" * 100)

print("""
Advantages:
✓ Simple implementation
✓ Preserves intra-class diversity
✓ Scalable

Limitations for this project:
✗ Dataset sizes are highly unequal
✗ Class distributions are highly unequal
✗ Sample reuse may become uncontrolled
✗ Large modalities can indirectly dominate training
✗ Pair counts may become unbalanced

Conclusion:
Random same-class pairing alone does not provide sufficient control
for the substantial modality and class imbalance present in this project.
""")

print("=" * 100)

STRATEGY A — RANDOM SAME-CLASS PAIRING ANALYSIS


,Modality,Total_Samples,NON_ASD_Available,ASD_Available
0,Genetic,116,29,87
1,Behavioral,880,439,441
2,Environmental,47768,45650,2118



----------------------------------------------------------------------------------------------------
SCIENTIFIC ASSESSMENT
----------------------------------------------------------------------------------------------------

Advantages:
✓ Simple implementation
✓ Preserves intra-class diversity
✓ Scalable

Limitations for this project:
✗ Dataset sizes are highly unequal
✗ Class distributions are highly unequal
✗ Sample reuse may become uncontrolled
✗ Large modalities can indirectly dominate training
✗ Pair counts may become unbalanced

Conclusion:
Random same-class pairing alone does not provide sufficient control
for the substantial modality and class imbalance present in this project.



In [7]:
# =============================================================================
# CELL 7 — ANALYZE BALANCED CLASS-CONDITIONAL PAIRING
# =============================================================================

print("=" * 100)
print("STRATEGY B — BALANCED CLASS-CONDITIONAL PAIRING")
print("=" * 100)

# -------------------------------------------------------------------------
# Determine the maximum common availability per class
# -------------------------------------------------------------------------

class_availability = {}

for modality, data in modalities.items():

    labels = data["labels"]

    unique, counts = np.unique(labels, return_counts=True)

    class_availability[modality] = dict(zip(unique, counts))


common_class_counts = {}

for class_label in [0, 1]:

    available_counts = [
        class_availability[modality].get(class_label, 0)
        for modality in modalities.keys()
    ]

    common_class_counts[class_label] = min(available_counts)


print("\nCOMMON CLASS AVAILABILITY ACROSS MODALITIES")
print("-" * 80)

for class_label, count in common_class_counts.items():

    class_name = "NON-ASD" if class_label == 0 else "ASD"

    print(f"{class_name}: {count} samples available in every modality")


balanced_pairing_summary = []

for modality in modalities.keys():

    balanced_pairing_summary.append({
        "Modality": modality,
        "NON_ASD_Selected": common_class_counts[0],
        "ASD_Selected": common_class_counts[1],
        "Total_Selected":
            common_class_counts[0] + common_class_counts[1]
    })


balanced_pairing_df = pd.DataFrame(balanced_pairing_summary)

print("\nBALANCED CONTRIBUTION")
display(balanced_pairing_df)


print("\n" + "-" * 100)
print("SCIENTIFIC ASSESSMENT")
print("-" * 100)

print("""
Advantages:
✓ Prevents large modality domination
✓ Controls class contribution
✓ Preserves individual representation diversity
✓ Compatible with same-class semantic supervision
✓ Compatible with different-class negative relationships
✓ Scalable
✓ Reproducible using fixed random seeds
✓ Does not require patient correspondence

Potential Limitation:
• Requires careful sampling and pair metadata tracking

Conclusion:
Highly suitable for knowledge-level cross-modal learning under
substantial dataset and class imbalance.
""")

print("=" * 100)

STRATEGY B — BALANCED CLASS-CONDITIONAL PAIRING

COMMON CLASS AVAILABILITY ACROSS MODALITIES
--------------------------------------------------------------------------------
NON-ASD: 29 samples available in every modality
ASD: 87 samples available in every modality

BALANCED CONTRIBUTION


,Modality,NON_ASD_Selected,ASD_Selected,Total_Selected
0,Genetic,29,87,116
1,Behavioral,29,87,116
2,Environmental,29,87,116



----------------------------------------------------------------------------------------------------
SCIENTIFIC ASSESSMENT
----------------------------------------------------------------------------------------------------

Advantages:
✓ Prevents large modality domination
✓ Controls class contribution
✓ Preserves individual representation diversity
✓ Compatible with same-class semantic supervision
✓ Compatible with different-class negative relationships
✓ Scalable
✓ Reproducible using fixed random seeds
✓ Does not require patient correspondence

Potential Limitation:
• Requires careful sampling and pair metadata tracking

Conclusion:
Highly suitable for knowledge-level cross-modal learning under
substantial dataset and class imbalance.



In [8]:
# =============================================================================
# CELL 8 — ANALYZE CENTROID / PROTOTYPE PAIRING
# =============================================================================

print("=" * 100)
print("STRATEGY C — CENTROID / PROTOTYPE-BASED PAIRING")
print("=" * 100)

centroid_records = []

for modality, data in modalities.items():

    embeddings = data["embeddings"]
    labels = data["labels"]

    for class_label in [0, 1]:

        class_embeddings = embeddings[labels == class_label]

        centroid = np.mean(class_embeddings, axis=0)

        centroid_records.append({
            "Modality": modality,
            "Class": (
                "NON-ASD"
                if class_label == 0
                else "ASD"
            ),
            "Samples_Represented": len(class_embeddings),
            "Centroid_Dimension": len(centroid)
        })


centroid_df = pd.DataFrame(centroid_records)

display(centroid_df)


total_centroids = len(centroid_df)

print("\n" + "-" * 100)
print("REPRESENTATION REDUCTION")
print("-" * 100)

print(f"Original total representations: {sum(len(data['labels']) for data in modalities.values())}")
print(f"Class prototypes produced: {total_centroids}")

print("\nSCIENTIFIC ASSESSMENT")
print("-" * 100)

print("""
Advantages:
✓ Strongly reduces dataset imbalance
✓ Provides interpretable class-level knowledge prototypes
✓ Computationally efficient
✓ No patient matching required

Limitations:
✗ Collapses within-class diversity
✗ Produces very limited training relationships
✗ Cannot capture heterogeneous representation structures
✗ May oversimplify complex modality knowledge

Conclusion:
Centroid representations are valuable for analysis and auxiliary
regularization but are likely too information-compressed to serve as
the sole cross-modal training pairing strategy.
""")

print("=" * 100)

STRATEGY C — CENTROID / PROTOTYPE-BASED PAIRING


,Modality,Class,Samples_Represented,Centroid_Dimension
0,Genetic,NON-ASD,29,32
1,Genetic,ASD,87,32
2,Behavioral,NON-ASD,439,32
3,Behavioral,ASD,441,32
4,Environmental,NON-ASD,45650,32
5,Environmental,ASD,2118,32



----------------------------------------------------------------------------------------------------
REPRESENTATION REDUCTION
----------------------------------------------------------------------------------------------------
Original total representations: 48764
Class prototypes produced: 6

SCIENTIFIC ASSESSMENT
----------------------------------------------------------------------------------------------------

Advantages:
✓ Strongly reduces dataset imbalance
✓ Provides interpretable class-level knowledge prototypes
✓ Computationally efficient
✓ No patient matching required

Limitations:
✗ Collapses within-class diversity
✗ Produces very limited training relationships
✗ Cannot capture heterogeneous representation structures
✗ May oversimplify complex modality knowledge

Conclusion:
Centroid representations are valuable for analysis and auxiliary
regularization but are likely too information-compressed to serve as
the sole cross-modal training pairing strategy.



In [9]:
# =============================================================================
# CELL 9 — ANALYZE SIMILARITY / DISTRIBUTION-AWARE PAIRING
# =============================================================================

print("=" * 100)
print("STRATEGY D — SIMILARITY / DISTRIBUTION-AWARE PAIRING")
print("=" * 100)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


# -------------------------------------------------------------------------
# Create balanced diagnostic samples
# -------------------------------------------------------------------------

def balanced_sample(embeddings, labels, n_per_class, rng):

    selected_indices = []

    for class_label in [0, 1]:

        class_indices = np.where(labels == class_label)[0]

        selected = rng.choice(
            class_indices,
            size=n_per_class[class_label],
            replace=False
        )

        selected_indices.extend(selected)

    selected_indices = np.array(selected_indices)

    return embeddings[selected_indices], labels[selected_indices]


n_per_class = {
    0: common_class_counts[0],
    1: common_class_counts[1]
}


balanced_data = {}

for modality, data in modalities.items():

    emb, lab = balanced_sample(
        data["embeddings"],
        data["labels"],
        n_per_class,
        rng
    )

    balanced_data[modality] = {
        "embeddings": emb,
        "labels": lab
    }


# -------------------------------------------------------------------------
# Compare existing cross-modal geometric similarity
# -------------------------------------------------------------------------

modality_pairs = [
    ("Genetic", "Behavioral"),
    ("Genetic", "Environmental"),
    ("Behavioral", "Environmental")
]

similarity_diagnostics = []

for modality_a, modality_b in modality_pairs:

    emb_a = balanced_data[modality_a]["embeddings"]
    emb_b = balanced_data[modality_b]["embeddings"]

    similarity_matrix = cosine_similarity(emb_a, emb_b)

    similarity_diagnostics.append({
        "Modality_Pair": f"{modality_a} vs {modality_b}",
        "Mean_Cosine_Similarity": float(np.mean(similarity_matrix)),
        "Max_Cosine_Similarity": float(np.max(similarity_matrix)),
        "Min_Cosine_Similarity": float(np.min(similarity_matrix))
    })


similarity_diagnostics_df = pd.DataFrame(similarity_diagnostics)

display(similarity_diagnostics_df)


print("\n" + "-" * 100)
print("SCIENTIFIC ASSESSMENT")
print("-" * 100)

print("""
Similarity-aware pairing can identify representations that are already
geometrically close.

However, Phase 01 demonstrated that natural cross-modal compatibility
is partial and modality-dependent.

Therefore, using existing similarity as the PRIMARY pairing criterion
could introduce selection bias:

The model may preferentially learn relationships that are already easy
to align while underrepresenting the broader knowledge distribution.

Conclusion:
Similarity information is useful as a diagnostic or auxiliary analysis,
but should not be the sole criterion for defining semantic cross-modal
relationships in this project.
""")

print("=" * 100)

STRATEGY D — SIMILARITY / DISTRIBUTION-AWARE PAIRING


,Modality_Pair,Mean_Cosine_Similarity,Max_Cosine_Similarity,Min_Cosine_Similarity
0,Genetic vs Behavioral,-0.000528,0.564816,-0.619270
1,Genetic vs Environmental,0.000008,0.598995,-0.614331
2,Behavioral vs Environmental,0.010361,0.642695,-0.558085



----------------------------------------------------------------------------------------------------
SCIENTIFIC ASSESSMENT
----------------------------------------------------------------------------------------------------

Similarity-aware pairing can identify representations that are already
geometrically close.

However, Phase 01 demonstrated that natural cross-modal compatibility
is partial and modality-dependent.

Therefore, using existing similarity as the PRIMARY pairing criterion
could introduce selection bias:

The model may preferentially learn relationships that are already easy
to align while underrepresenting the broader knowledge distribution.

Conclusion:
Similarity information is useful as a diagnostic or auxiliary analysis,
but should not be the sole criterion for defining semantic cross-modal
relationships in this project.



In [10]:
# =============================================================================
# CELL 10 — FORMAL PAIRING STRATEGY EVALUATION MATRIX
# =============================================================================

print("=" * 100)
print("FORMAL PAIRING STRATEGY EVALUATION")
print("=" * 100)

strategy_evaluation = pd.DataFrame([

    {
        "Strategy": "Random Same-Class Pairing",
        "Knowledge_Level_Validity": "High",
        "Dataset_Imbalance_Control": "Low",
        "Class_Imbalance_Control": "Low",
        "Preserves_Representation_Diversity": "High",
        "Avoids_Similarity_Selection_Bias": "High",
        "Scalability": "High",
        "Reproducibility": "Moderate",
        "Overall_Suitability": "Moderate"
    },

    {
        "Strategy": "Balanced Class-Conditional Pairing",
        "Knowledge_Level_Validity": "High",
        "Dataset_Imbalance_Control": "High",
        "Class_Imbalance_Control": "High",
        "Preserves_Representation_Diversity": "High",
        "Avoids_Similarity_Selection_Bias": "High",
        "Scalability": "High",
        "Reproducibility": "High",
        "Overall_Suitability": "High"
    },

    {
        "Strategy": "Centroid / Prototype Pairing",
        "Knowledge_Level_Validity": "High",
        "Dataset_Imbalance_Control": "High",
        "Class_Imbalance_Control": "High",
        "Preserves_Representation_Diversity": "Low",
        "Avoids_Similarity_Selection_Bias": "High",
        "Scalability": "High",
        "Reproducibility": "High",
        "Overall_Suitability": "Moderate"
    },

    {
        "Strategy": "Similarity / Distribution-Aware Pairing",
        "Knowledge_Level_Validity": "Moderate",
        "Dataset_Imbalance_Control": "Requires Additional Control",
        "Class_Imbalance_Control": "Requires Additional Control",
        "Preserves_Representation_Diversity": "High",
        "Avoids_Similarity_Selection_Bias": "Low",
        "Scalability": "Moderate",
        "Reproducibility": "Moderate",
        "Overall_Suitability": "Partial"
    }

])

display(strategy_evaluation)

print("\n" + "=" * 100)
print("EVALUATION COMPLETED")
print("=" * 100)

FORMAL PAIRING STRATEGY EVALUATION


,Strategy,Knowledge_Level_Validity,Dataset_Imbalance_Control,Class_Imbalance_Control,Preserves_Representation_Diversity,Avoids_Similarity_Selection_Bias,Scalability,Reproducibility,Overall_Suitability
0,Random Same-Class Pairing,High,Low,Low,High,High,High,Moderate,Moderate
1,Balanced Class-Conditional Pairing,High,High,High,High,High,High,High,High
2,Centroid / Prototype Pairing,High,High,High,Low,High,High,High,Moderate
3,Similarity / Distribution-Aware Pairing,Moderate,Requires Additional Control,Requires Additional Control,High,Low,Moderate,Moderate,Partial



EVALUATION COMPLETED


In [11]:
# =============================================================================
# CELL 11 — QUANTITATIVE STRATEGY SUITABILITY SCORING
# =============================================================================

print("=" * 100)
print("PAIRING STRATEGY SUITABILITY SCORING")
print("=" * 100)

strategy_scores = pd.DataFrame([

    {
        "Strategy": "Random Same-Class Pairing",
        "Knowledge_Level_Validity": 3,
        "Imbalance_Control": 1,
        "Diversity_Preservation": 3,
        "Bias_Avoidance": 3,
        "Scalability": 3,
        "Reproducibility": 2
    },

    {
        "Strategy": "Balanced Class-Conditional Pairing",
        "Knowledge_Level_Validity": 3,
        "Imbalance_Control": 3,
        "Diversity_Preservation": 3,
        "Bias_Avoidance": 3,
        "Scalability": 3,
        "Reproducibility": 3
    },

    {
        "Strategy": "Centroid / Prototype Pairing",
        "Knowledge_Level_Validity": 3,
        "Imbalance_Control": 3,
        "Diversity_Preservation": 1,
        "Bias_Avoidance": 3,
        "Scalability": 3,
        "Reproducibility": 3
    },

    {
        "Strategy": "Similarity / Distribution-Aware Pairing",
        "Knowledge_Level_Validity": 2,
        "Imbalance_Control": 1,
        "Diversity_Preservation": 3,
        "Bias_Avoidance": 1,
        "Scalability": 2,
        "Reproducibility": 2
    }

])

score_columns = [
    "Knowledge_Level_Validity",
    "Imbalance_Control",
    "Diversity_Preservation",
    "Bias_Avoidance",
    "Scalability",
    "Reproducibility"
]

strategy_scores["Total_Score"] = (
    strategy_scores[score_columns].sum(axis=1)
)

strategy_scores = strategy_scores.sort_values(
    "Total_Score",
    ascending=False
).reset_index(drop=True)

display(strategy_scores)

print("\n" + "-" * 100)
print("SCORING NOTE")
print("-" * 100)

print("""
This scoring framework is a structured methodological comparison.

It does not independently prove scientific superiority.

The final strategy decision must consider:

• Phase 01 embedding analysis
• Dataset size differences
• Class imbalance
• Lack of patient correspondence
• Representation diversity
• Risk of geometric selection bias

The score is therefore supporting evidence, not the only evidence.
""")

print("=" * 100)

PAIRING STRATEGY SUITABILITY SCORING


,Strategy,Knowledge_Level_Validity,Imbalance_Control,Diversity_Preservation,Bias_Avoidance,Scalability,Reproducibility,Total_Score
0,Balanced Class-Conditional Pairing,3,3,3,3,3,3,18
1,Centroid / Prototype Pairing,3,3,1,3,3,3,16
2,Random Same-Class Pairing,3,1,3,3,3,2,15
3,Similarity / Distribution-Aware Pairing,2,1,3,1,2,2,11



----------------------------------------------------------------------------------------------------
SCORING NOTE
----------------------------------------------------------------------------------------------------

This scoring framework is a structured methodological comparison.

It does not independently prove scientific superiority.

The final strategy decision must consider:

• Phase 01 embedding analysis
• Dataset size differences
• Class imbalance
• Lack of patient correspondence
• Representation diversity
• Risk of geometric selection bias

The score is therefore supporting evidence, not the only evidence.



In [12]:
# =============================================================================
# CELL 12 — FINAL PAIRING STRATEGY COMPARISON CONCLUSION
# =============================================================================

print("=" * 100)
print("STEP 02.2 — FINAL PAIRING STRATEGY COMPARISON CONCLUSION")
print("=" * 100)

best_strategy = strategy_scores.iloc[0]["Strategy"]
best_score = strategy_scores.iloc[0]["Total_Score"]

print("\nEVALUATED STRATEGIES:")
for _, row in strategy_scores.iterrows():
    print(
        f"  {row['Strategy']}: "
        f"Suitability Score = {row['Total_Score']}"
    )


print("\n" + "-" * 100)
print("COMPARATIVE CONCLUSION")
print("-" * 100)

print(f"""
Based on the structured comparison, the strongest candidate strategy is:

{best_strategy}

This conclusion is supported by the characteristics of the current
cross-modal ASD datasets:

1. The modalities have substantially different dataset sizes.

2. Environmental data has extreme class imbalance.

3. No shared patient identities exist across modalities.

4. Individual embedding diversity should be preserved.

5. Natural cross-modal alignment is partial rather than complete.

6. The methodology requires knowledge-level rather than patient-level
   relationships.

Balanced class-conditional pairing addresses these constraints by
controlling both modality and class contribution while preserving
individual representation diversity.

Random pairing lacks sufficient imbalance control.

Centroid pairing is useful for aggregate analysis but compresses
within-class diversity.

Similarity-aware pairing may introduce geometric selection bias because
the model would preferentially learn relationships that are already
naturally close.

Therefore, balanced class-conditional pairing emerges as the strongest
candidate for the next stage of formal strategy selection.
""")

print("\nIMPORTANT:")
print("""
This notebook compares strategies.

The final methodological selection will be formally documented and
justified in:

STEP 02.3 — FINAL PAIRING STRATEGY SELECTION
""")

print("=" * 100)
print("✓ STEP 02.2 COMPLETED")
print("✓ MULTIPLE PAIRING STRATEGIES COMPARED")
print("✓ DATASET IMBALANCE CONSIDERED")
print("✓ REPRESENTATION DIVERSITY CONSIDERED")
print("✓ KNOWLEDGE-LEVEL METHODOLOGY PRESERVED")
print("✓ NO PATIENT-LEVEL MATCHING USED")
print("=" * 100)

STEP 02.2 — FINAL PAIRING STRATEGY COMPARISON CONCLUSION

EVALUATED STRATEGIES:
  Balanced Class-Conditional Pairing: Suitability Score = 18
  Centroid / Prototype Pairing: Suitability Score = 16
  Random Same-Class Pairing: Suitability Score = 15
  Similarity / Distribution-Aware Pairing: Suitability Score = 11

----------------------------------------------------------------------------------------------------
COMPARATIVE CONCLUSION
----------------------------------------------------------------------------------------------------

Based on the structured comparison, the strongest candidate strategy is:

Balanced Class-Conditional Pairing

This conclusion is supported by the characteristics of the current
cross-modal ASD datasets:

1. The modalities have substantially different dataset sizes.

2. Environmental data has extreme class imbalance.

3. No shared patient identities exist across modalities.

4. Individual embedding diversity should be preserved.

5. Natural cross-modal ali

In [13]:
# =============================================================================
# CELL 13 — SAVE STRATEGY COMPARISON ARTIFACTS
# =============================================================================

print("=" * 100)
print("SAVING PAIRING STRATEGY COMPARISON ARTIFACTS")
print("=" * 100)

# Dataset characteristics
dataset_analysis_path = (
    REPORT_DIR
    / "02_02_dataset_pairing_characteristics.csv"
)

dataset_analysis_df.to_csv(
    dataset_analysis_path,
    index=False
)


# Strategy definitions
strategy_definitions_path = (
    REPORT_DIR
    / "02_02_pairing_strategy_definitions.csv"
)

strategy_definitions.to_csv(
    strategy_definitions_path,
    index=False
)


# Balanced pairing summary
balanced_pairing_path = (
    REPORT_DIR
    / "02_02_balanced_pairing_availability.csv"
)

balanced_pairing_df.to_csv(
    balanced_pairing_path,
    index=False
)


# Similarity diagnostics
similarity_diagnostics_path = (
    REPORT_DIR
    / "02_02_similarity_pairing_diagnostics.csv"
)

similarity_diagnostics_df.to_csv(
    similarity_diagnostics_path,
    index=False
)


# Formal evaluation
evaluation_path = (
    REPORT_DIR
    / "02_02_strategy_evaluation_matrix.csv"
)

strategy_evaluation.to_csv(
    evaluation_path,
    index=False
)


# Quantitative scoring
scoring_path = (
    REPORT_DIR
    / "02_02_strategy_suitability_scoring.csv"
)

strategy_scores.to_csv(
    scoring_path,
    index=False
)


print("\n✓ Dataset characteristics:")
print(dataset_analysis_path)

print("\n✓ Strategy definitions:")
print(strategy_definitions_path)

print("\n✓ Balanced pairing availability:")
print(balanced_pairing_path)

print("\n✓ Similarity diagnostics:")
print(similarity_diagnostics_path)

print("\n✓ Evaluation matrix:")
print(evaluation_path)

print("\n✓ Strategy scoring:")
print(scoring_path)


print("\n" + "=" * 100)
print("STEP 02.2 — ARTIFACT SAVING COMPLETED")
print("=" * 100)

SAVING PAIRING STRATEGY COMPARISON ARTIFACTS

✓ Dataset characteristics:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_02_dataset_pairing_characteristics.csv

✓ Strategy definitions:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_02_pairing_strategy_definitions.csv

✓ Balanced pairing availability:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_02_balanced_pairing_availability.csv

✓ Similarity diagnostics:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_02_similarity_pairing_diagnostics.csv

✓ Evaluation matrix:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_02_strategy_evaluation_matrix